# Regression Trees, Random Forest and Gradient Boosting – Cycling at the Lake of Zurich

**Dataset:** Bike counts in Zurich 2024–2025: daily number of bikes at 16 automatic counting stations of the city of
Zurich, with weather and calendar information (Open Data Stadt Zürich). File: `../00_Data/zurich_bike_counts_2024_2025.csv`.
We model the counting station **Mythenquai** – a cycle route along the lake between the city centre and Wollishofen,
used both by commuters and for leisure.

**Business question:** How many cyclists will pass the Mythenquai on a given day, depending on weather, weekday and
holidays? Weather and weekday interact (a sunny Sunday attracts many leisure cyclists, a sunny Tuesday mainly commuters)
– a typical use case for tree-based models.

**After this exercise you can ...**
- find the best split of a regression tree by hand (SSE criterion),
- fit, visualise and prune a regression tree,
- explain and apply bagging, random forests (incl. out-of-bag error, permutation importance) and gradient boosting,
- compare models fairly with cross-validation.

## Concept

The concepts behind this exercise (formulas, worked examples and additional explanations) are
explained in the notebook [regression_trees_ensembles_concept.ipynb](regression_trees_ensembles_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    KFold,
)
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Best split of a regression tree.** On six days the following numbers of bikes (in hundreds) were counted at the
Mythenquai:

| daily mean temperature $x$ [°C] | −2 | 3 | 8 | 14 | 20 | 26 |
|---|---|---|---|---|---|---|
| bikes $y$ [hundreds] | 15 | 22 | 30 | 38 | 42 | 44 |

Compute the SSE of the root node. Then compute, for each of the five possible thresholds (midpoints between neighbouring
temperatures), the means and the SSE of the two child nodes. Which split does the tree choose? What are the leaf
predictions?

**b) Variance of an average.** 100 trees have variance $\sigma^2 = 4$ each. Compute the variance of their average for
$\rho = 0.6$ (bagged trees, similar to each other) and for $\rho = 0.2$ (random forest). What happens for
$B \to \infty$?

**c) One boosting step.** The current prediction for a day is 30 (hundred bikes), the observed number is 40. The next
tree predicts the residual exactly. What is the new prediction with learning rate $\nu = 0.1$? How many such steps are
needed until the remaining residual is below 5?

**d)** Verify your results with Python.

In [ ]:
# a) Best split: temperature x and bikes y (in hundreds)
x = np.array([-2, 3, 8, 14, 20, 26])
y = np.array([15, 22, 30, 38, 42, 44])

# Your code here

# b) Variance of the average of 100 trees with variance 4
# Your code here

# c) Boosting steps until the residual is below 5
# Your code here

**Your answer:** ...

## Exercise 2: Import and explore the bike counts at the Mythenquai

a) Import the bike counts. The target is the column `mythenquai` (bikes per day at this station). Use the features
`temp_mean`, `temp_max`, `rain_duration_min`, `solar_radiation`, `month`, `holiday`, `school_holiday` and the
`weekday` (as dummy variables: `pd.get_dummies(..., columns=['weekday'], dtype=int)`).

b) Plot `mythenquai` against `temp_max`, separately for working days and weekends. What do you see?

c) Split the data into training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data as bikes
#    File: '../00_Data/zurich_bike_counts_2024_2025.csv'
# Your code here

# Features (weekday as dummy variables: one 0/1 column per weekday)
features = [
    'temp_mean',
    'temp_max',
    'rain_duration_min',
    'solar_radiation',
    'month',
    'holiday',
    'school_holiday',
    'weekday',
]

X = ___
y = bikes['mythenquai']

# b) Scatter plot for working days and weekends
# Your code here

# c) Train/test split
X_train, X_test, y_train, y_test = ___

**Your answer:** ...

## Exercise 3: Baseline – linear regression

Fit a multiple linear regression with all features on the training set and compute RMSE and $R^2$ on the test set. This
is the baseline that the tree-based models have to beat.

In [ ]:
# Exercise 3: linear regression with all features; test RMSE and R²
# Your code here

## Exercise 4: A small regression tree

a) Fit a `DecisionTreeRegressor(max_depth=3, random_state=42)` on the training set.

b) Visualise the tree with `tree.plot_tree()` and interpret the first two levels. Which features are used?

c) Compute RMSE and $R^2$ on the test set. How does the small tree compare to the linear regression?

In [ ]:
# a) Fit the tree
reg_tree = ___

# b) Plot the tree
# Your code here

# c) Test RMSE and R²
# Your code here

**Your answer:** ...

## Exercise 5: Tree complexity and pruning

a) For `max_depth` = 1, 2, …, 15 compute the RMSE on the training set and the 5-fold CV RMSE (on the training set).
Plot both curves. Where does the tree start to overfit?

b) Alternatively, grow a full tree and prune it: compute the candidate values with
`DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train).ccp_alphas` and choose `ccp_alpha`
with `GridSearchCV` (5-fold CV, `scoring='neg_root_mean_squared_error'`). How many leaves does the best pruned tree have?
What is its test RMSE?

In [ ]:
# Exercise 5: tree complexity and pruning
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# a) Training and CV error for different depths
depths = range(1, 16)
rmse_train = []
rmse_cv = []
for d in depths:
    # Your code here
    pass

# Plot of both errors
# Your code here

# b) Pruning: candidate alphas of the full tree, GridSearchCV for ccp_alpha
#    (store the best value as best_alpha)
# Your code here

**Your answer:** ...

## Exercise 6: Random forest

a) Fit random forests with `n_estimators` = 1, 5, 10, 25, 50, 100, 200, 400 (`oob_score=True`, `random_state=42`,
`n_jobs=-1`). The OOB predictions are stored in `oob_prediction_`. Plot the OOB RMSE against the number of trees. When do
the gains plateau?

b) Compare `max_features` = 1.0 (all features, i.e. bagging), `'sqrt'` and 0.5 with 5-fold CV (300 trees).

c) For the best forest, compute the test RMSE. Compare the impurity-based feature importance (`feature_importances_`)
with the **permutation importance** on the test set.

In [ ]:
# a) Number of trees vs. OOB error
n_trees = [1, 5, 10, 25, 50, 100, 200, 400]
oob_rmse = []
for n in n_trees:
    rf = RandomForestRegressor(
        n_estimators=n, oob_score=True, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    # OOB prediction of each training day (NaN if the day was never OOB)
    oob_pred = rf.oob_prediction_
    ok = ~np.isnan(oob_pred)
    # Your code here

# Plot of the OOB RMSE
# Your code here

# b) 5-fold CV RMSE for different values of max_features
# Your code here

In [ ]:
# c) Random forest with the best setting from b): test RMSE and feature
#    importance (impurity-based and permutation importance)
# Your code here

**Your answer:** ...

## Exercise 7: Gradient boosting

a) Fit `GradientBoostingRegressor(n_estimators=1000, max_depth=3, random_state=42)` with learning rates
$\nu$ = 0.01, 0.1 and 1.0. Use `staged_predict(X_test)` to compute the test RMSE after each boosting iteration and plot
the three curves.

*Note: we look at the test curve here only for illustration. In practice, `learning_rate` and `n_estimators` are tuned
with cross-validation (see Exercise 8).*

b) Describe the influence of the learning rate. What happens with $\nu = 1.0$?

In [ ]:
# a) Test error after each tree for three learning rates
for lr in [0.01, 0.1, 1.0]:
    gb = ___

    # staged_predict gives the predictions after 1, 2, ..., 1000 trees
    rmse_staged = []
    for y_pred in gb.staged_predict(X_test):
        rmse_staged.append(root_mean_squared_error(y_test, y_pred))
    plt.plot(range(1, 1001), rmse_staged, label='learning rate ' + str(lr))

# Figure showing the test error by number of trees
plt.ylim(0, 1000)
plt.xlabel('Number of trees M')
plt.ylabel('Test RMSE [bikes per day]')
plt.legend()
plt.show()

**Your answer:** ...

## Exercise 8: Fair model comparison

Compare the following models with **5-fold CV RMSE on the training set** and summarise the results in a table:
linear regression, pruned tree (best `ccp_alpha` from Exercise 5), random forest (best setting from Exercise 6) and
gradient boosting (`learning_rate=0.05`, `n_estimators=300`, `max_depth=3`). Then evaluate the best model **once** on the
test set. Which model would you recommend to the city, and what are the drawbacks?

In [ ]:
# Exercise 8: the four models (settings from Exercises 5 and 6)
models = {
    'Linear regression': LinearRegression(),
    'Pruned tree': ___,
    'Random forest': ___,
    'Gradient boosting': ___,
}

# 5-fold CV RMSE of all models (store them in cv_rmse)
cv_rmse = {}
for name, model in models.items():
    # Your code here
    pass

# Fit the best model and evaluate it once on the test set
# Your code here

**Your answer:** ...

## Exercise 9: More covariates

So far the models used the weather (temperature, rain, sun) and the calendar (month, weekday, holidays). The data set
contains more information: the air pressure (`pressure`), the exact position in the year (`day_of_year`, computed from
`date`) and the `year` (2024 or 2025).

a) Add these three covariates and split the data again (80/20, `random_state=42` – the same days as before).

b) Compute the 5-fold CV RMSE of the linear regression, the random forest (best setting of Exercise 6) and gradient
boosting (as in Exercise 8) with the **extended** feature set and compare with Exercise 8.

c) Evaluate the best model on the test set and compute its permutation importance. Which new covariate matters most, and
why?

In [ ]:
# a) Extended feature set
bikes['day_of_year'] = pd.to_datetime(bikes['date']).dt.dayofyear
features_more = features + ['pressure', 'day_of_year', 'year']
X_more = ___

# Same random_state as before -> the same training and test days
X2_train, X2_test, y2_train, y2_test = ___

# b) CV RMSE with the extended feature set, compared with Exercise 8
# Your code here

# c) Gradient boosting with more covariates: test RMSE and permutation
#    importance
# Your code here

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')